<a href="https://colab.research.google.com/github/bertramwooster/PythonDataScienceHandbook/blob/new_branch/notebooks/03.08-Aggregation-and-Grouping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aggregation and Grouping

A fundamental piece of many data analysis tasks is efficient summarization: computing aggregations like `sum`, `mean`, `median`, `min`, and `max`, in which a single number summarizes aspects of a potentially large dataset.
In this chapter, we'll explore aggregations in Pandas, from simple operations akin to what we've seen on NumPy arrays to more sophisticated operations based on the concept of a `groupby`.

For convenience, we'll use the same `display` magic function that we used in the previous chapters:

In [1]:
import numpy as np
import pandas as pd

class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args

    def _repr_html_(self):
        return '\n'.join(self.template.format(a, eval(a)._repr_html_())
                         for a in self.args)

    def __repr__(self):
        return '\n\n'.join(a + '\n' + repr(eval(a))
                           for a in self.args)

## Planets Data

Here we will use the Planets dataset, available via the [Seaborn package](http://seaborn.pydata.org/) (see [Visualization With Seaborn](04.14-Visualization-With-Seaborn.ipynb)).
It gives information on planets that astronomers have discovered around other stars (known as *extrasolar planets*, or *exoplanets* for short). It can be downloaded with a simple Seaborn command:

In [2]:
import seaborn as sns
planets = sns.load_dataset('planets')
planets.shape

(1035, 6)

In [3]:
planets.head()

,method,number,orbital_period,mass,distance,year
0,Radial Velocity,1,269.300,7.10,77.40,2006
1,Radial Velocity,1,874.774,2.21,56.95,2008
2,Radial Velocity,1,763.000,2.60,19.84,2011
3,Radial Velocity,1,326.030,19.40,110.62,2007
4,Radial Velocity,1,516.220,10.50,119.47,2009


In [4]:
planets['number'].unique()

array([1, 2, 3, 5, 4, 6, 7])

In [5]:
[planets[planets['number']==i].shape[0] for i in planets['number'].unique()]

[595, 259, 88, 30, 32, 24, 7]

This has some details on the 1,000+ extrasolar planets discovered up to 2014.

## Simple Aggregation in Pandas

In ["Aggregations: Min, Max, and Everything In Between"](02.04-Computation-on-arrays-aggregates.ipynb), we explored some of the data aggregations available for NumPy arrays.
As with a one-dimensional NumPy array, for a Pandas ``Series`` the aggregates return a single value:

In [8]:
rng = np.random.default_rng(42)
ser = pd.Series(rng.random(5))
ser

,0
0,0.773956
1,0.438878
2,0.858598
3,0.697368
4,0.094177


In [9]:
ser.sum()

np.float64(2.8629777851664118)

In [10]:
ser.mean()

np.float64(0.5725955570332824)

For a `DataFrame`, by default the aggregates return results within each column:

In [11]:
df = pd.DataFrame({'A': rng.random(5),
                   'B': rng.random(5)})
df

,A,B
0,0.975622,0.370798
1,0.761140,0.926765
2,0.786064,0.643865
3,0.128114,0.822762
4,0.450386,0.443414


In [12]:
df.mean()

,0
A,0.620265
B,0.641521


By specifying the `axis` argument, you can instead aggregate within each row:

In [13]:
df.mean(axis='columns')

,0
0,0.673210
1,0.843952
2,0.714965
3,0.475438
4,0.446900


Pandas `Series` and `DataFrame` objects include all of the common aggregates mentioned in [Aggregations: Min, Max, and Everything In Between](02.04-Computation-on-arrays-aggregates.ipynb); in addition, there is a convenience method, `describe`, that computes several common aggregates for each column and returns the result.
Let's use this on the Planets data, for now dropping rows with missing values:

In [14]:
planets.dropna().describe()

,number,orbital_period,mass,distance,year
count,498.00000,498.000000,498.000000,498.000000,498.000000
mean,1.73494,835.778671,2.509320,52.068213,2007.377510
std,1.17572,1469.128259,3.636274,46.596041,4.167284
min,1.00000,1.328300,0.003600,1.350000,1989.000000
25%,1.00000,38.272250,0.212500,24.497500,2005.000000
50%,1.00000,357.000000,1.245000,39.940000,2009.000000
75%,2.00000,999.600000,2.867500,59.332500,2011.000000
max,6.00000,17337.500000,25.000000,354.000000,2014.000000


This method helps us understand the overall properties of a dataset.
For example, we see in the `year` column that although exoplanets were discovered as far back as 1989, half of all planets in the dataset were not discovered until 2010 or after.
This is largely thanks to the *Kepler* mission, which aimed to find eclipsing planets around other stars using a specially designed space telescope.

The following table summarizes some other built-in Pandas aggregations:

| Aggregation              | Returns                         |
|--------------------------|---------------------------------|
| ``count``                | Total number of items           |
| ``first``, ``last``      | First and last item             |
| ``mean``, ``median``     | Mean and median                 |
| ``min``, ``max``         | Minimum and maximum             |
| ``std``, ``var``         | Standard deviation and variance |
| ``mad``                  | Mean absolute deviation         |
| ``prod``                 | Product of all items            |
| ``sum``                  | Sum of all items                |

These are all methods of `DataFrame` and `Series` objects.

To go deeper into the data, however, simple aggregates are often not enough.
The next level of data summarization is the `groupby` operation, which allows you to quickly and efficiently compute aggregates on subsets of data.

## groupby: Split, Apply, Combine

Simple aggregations can give you a flavor of your dataset, but often we would prefer to aggregate conditionally on some label or index: this is implemented in the so-called `groupby` operation.
The name "group by" comes from a command in the SQL database language, but it is perhaps more illuminative to think of it in the terms first coined by Hadley Wickham of Rstats fame: *split, apply, combine*.

### Split, Apply, Combine

A canonical example of this split-apply-combine operation, where the "apply" is a summation aggregation, is illustrated in this figure:

![](https://github.com/bertramwooster/PythonDataScienceHandbook/blob/new_branch/notebooks/images/03.08-split-apply-combine.png?raw=1)

([figure source in Appendix](https://github.com/jakevdp/PythonDataScienceHandbook/blob/master/notebooks/06.00-Figure-Code.ipynb#Split-Apply-Combine))

This illustrates what the `groupby` operation accomplishes:

- The *split* step involves breaking up and grouping a `DataFrame` depending on the value of the specified key.
- The *apply* step involves computing some function, usually an aggregate, transformation, or filtering, within the individual groups.
- The *combine* step merges the results of these operations into an output array.

While this could certainly be done manually using some combination of the masking, aggregation, and merging commands covered earlier, an important realization is that *the intermediate splits do not need to be explicitly instantiated*. Rather, the `groupby` can (often) do this in a single pass over the data, updating the sum, mean, count, min, or other aggregate for each group along the way.
The power of the `groupby` is that it abstracts away these steps: the user need not think about *how* the computation is done under the hood, but rather can think about the *operation as a whole*.

As a concrete example, let's take a look at using Pandas for the computation shown in the following figure.
We'll start by creating the input `DataFrame`:

In [15]:
df = pd.DataFrame({'key': ['A', 'B', 'C', 'A', 'B', 'C'],
                   'data': range(6)}, columns=['key', 'data'])
df

,key,data
0,A,0
1,B,1
2,C,2
3,A,3
4,B,4
5,C,5


The most basic split-apply-combine operation can be computed with the `groupby` method of the `DataFrame`, passing the name of the desired key column:

In [16]:
df.groupby('key')

Notice that what is returned is a `DataFrameGroupBy` object, not a set of `DataFrame` objects.
This object is where the magic is: you can think of it as a special view of the `DataFrame`, which is poised to dig into the groups but does no actual computation until the aggregation is applied.
This "lazy evaluation" approach means that common aggregates can be implemented efficiently in a way that is almost transparent to the user.

To produce a result, we can apply an aggregate to this `DataFrameGroupBy` object, which will perform the appropriate apply/combine steps to produce the desired result:

In [17]:
df.groupby('key').sum()

,data
key,
A,3
B,5
C,7


The `sum` method is just one possibility here; you can apply most Pandas or NumPy aggregation functions, as well as most `DataFrame` operations, as you will see in the following discussion.

### The GroupBy Object

The `GroupBy` object is a flexible abstraction: in many ways, it can be treated as simply a collection of ``DataFrame``s, though it is doing more sophisticated things under the hood. Let's see some examples using the Planets data.

Perhaps the most important operations made available by a `GroupBy` are *aggregate*, *filter*, *transform*, and *apply*.
We'll discuss each of these more fully in the next section, but before that let's take a look at some of the other functionality that can be used with the basic `GroupBy` operation.

#### Column indexing

The `GroupBy` object supports column indexing in the same way as the `DataFrame`, and returns a modified `GroupBy` object.
For example:

In [18]:
planets.groupby('method')

In [19]:
planets.groupby('method')['orbital_period']

Here we've selected a particular `Series` group from the original `DataFrame` group by reference to its column name.
As with the `GroupBy` object, no computation is done until we call some aggregate on the object:

In [20]:
planets.groupby('method')['orbital_period'].median()

,orbital_period
method,
Astrometry,631.180000
Eclipse Timing Variations,4343.500000
Imaging,27500.000000
Microlensing,3300.000000
Orbital Brightness Modulation,0.342887
Pulsar Timing,66.541900
Pulsation Timing Variations,1170.000000
Radial Velocity,360.200000
Transit,5.714932


This gives an idea of the general scale of orbital periods (in days) that each method is sensitive to.

#### Iteration over groups

The `GroupBy` object supports direct iteration over the groups, returning each group as a `Series` or `DataFrame`:

In [21]:
for (method, group) in planets.groupby('method'):
    print("{0:30s} shape={1}".format(method, group.shape))

Astrometry                     shape=(2, 6)
Eclipse Timing Variations      shape=(9, 6)
Imaging                        shape=(38, 6)
Microlensing                   shape=(23, 6)
Orbital Brightness Modulation  shape=(3, 6)
Pulsar Timing                  shape=(5, 6)
Pulsation Timing Variations    shape=(1, 6)
Radial Velocity                shape=(553, 6)
Transit                        shape=(397, 6)
Transit Timing Variations      shape=(4, 6)


This can be useful for manual inspection of groups for the sake of debugging, but it is often much faster to use the built-in `apply` functionality, which we will discuss momentarily.

#### Dispatch methods

Through some Python class magic, any method not explicitly implemented by the `GroupBy` object will be passed through and called on the groups, whether they are `DataFrame` or `Series` objects.
For example, using the `describe` method is equivalent to calling `describe` on the `DataFrame` representing each group:

In [22]:
planets.groupby('method')['year'].describe()#.unstack()

,count,mean,std,min,25%,50%,75%,max
method,,,,,,,,
Astrometry,2.0,2011.500000,2.121320,2010.0,2010.75,2011.5,2012.25,2013.0
Eclipse Timing Variations,9.0,2010.000000,1.414214,2008.0,2009.00,2010.0,2011.00,2012.0
Imaging,38.0,2009.131579,2.781901,2004.0,2008.00,2009.0,2011.00,2013.0
Microlensing,23.0,2009.782609,2.859697,2004.0,2008.00,2010.0,2012.00,2013.0
Orbital Brightness Modulation,3.0,2011.666667,1.154701,2011.0,2011.00,2011.0,2012.00,2013.0
Pulsar Timing,5.0,1998.400000,8.384510,1992.0,1992.00,1994.0,2003.00,2011.0
Pulsation Timing Variations,1.0,2007.000000,NaN,2007.0,2007.00,2007.0,2007.00,2007.0
Radial Velocity,553.0,2007.518987,4.249052,1989.0,2005.00,2009.0,2011.00,2014.0
Transit,397.0,2011.236776,2.077867,2002.0,2010.00,2012.0,2013.00,2014.0


Looking at this table helps us to better understand the data: for example, the vast majority of planets until 2014 were discovered by the Radial Velocity and Transit methods, though the latter method became common more recently.
The newest methods seem to be Transit Timing Variation and Orbital Brightness Modulation, which were not used to discover a new planet until 2011.

Notice that these dispatch methods are applied *to each individual group*, and the results are then combined within `GroupBy` and returned.
Again, any valid `DataFrame`/`Series` method can be called in a similar manner on the corresponding `GroupBy` object.

### Aggregate, Filter, Transform, Apply

The preceding discussion focused on aggregation for the combine operation, but there are more options available.
In particular, `GroupBy` objects have `aggregate`, `filter`, `transform`, and `apply` methods that efficiently implement a variety of useful operations before combining the grouped data.

For the purpose of the following subsections, we'll use this ``DataFrame``:

In [41]:
import string
# Generate a random sequence of capital letters using random integers
def random_letters_from_integers(length = 10, rng = np.random.default_rng, k = 26):
    # ASCII values for capital letters 'A' to 'Z' are 65 to 90
    random_ascii_values = rng.integers(65, min(65 + k, 91), length)
    return [chr(value) for value in random_ascii_values]

# Example usage
rng = np.random.default_rng(0)
key = random_letters_from_integers(18, rng, 3)
print(key)

['C', 'B', 'B', 'A', 'A', 'A', 'A', 'A', 'A', 'C', 'B', 'C', 'B', 'B', 'C', 'C', 'B', 'B']


In [42]:

df = pd.DataFrame({'key': key,
                     'data1': range(len(key)),
                     'data2': rng.integers(0, 10, len(key))},
                    index = range(1,len(key) + 1),
                     columns = ['key', 'data1', 'data2'])
df



,key,data1,data2
1,C,0,5
2,B,1,9
3,B,2,2
4,A,3,8
5,A,4,6
6,A,5,0
7,A,6,3
8,A,7,8
9,A,8,5
10,C,9,0


#### Aggregation

You're now familiar with `GroupBy` aggregations with `sum`, `median`, and the like, but the `aggregate` method allows for even more flexibility.
It can take a string, a function, or a list thereof, and compute all the aggregates at once.
Here is a quick example combining all of these:

In [43]:
df.groupby('key').describe()

data1                                                    data2            \
    count       mean       std  min   25%   50%    75%   max count      mean   
key                                                                            
A     6.0   5.500000  1.870829  3.0  4.25   5.5   6.75   8.0   6.0  5.000000   
B     7.0  10.142857  6.362090  1.0  6.00  12.0  14.50  17.0   7.0  4.571429   
C     5.0   9.800000  5.974948  0.0  9.00  11.0  14.00  15.0   5.0  4.000000   

                                        
          std  min  25%  50%  75%  max  
key                                     
A    3.098387  0.0  3.5  5.5  7.5  8.0  
B    3.598942  0.0  1.5  5.0  7.5  9.0  
C    3.807887  0.0  0.0  5.0  7.0  8.0

In [44]:
df.groupby('key').aggregate(['min', 'median', 'max'])

data1            data2           
      min median max   min median max
key                                  
A       3    5.5   8     0    5.5   8
B       1   12.0  17     0    5.0   9
C       0   11.0  15     0    5.0   8

Another common pattern is to pass a dictionary mapping column names to operations to be applied on that column:

In [45]:
df.groupby('key').aggregate({'data1': 'min',
                             'data2': 'max'})

,data1,data2
key,,
A,3,8
B,1,9
C,0,8


#### Filtering

A filtering operation allows you to drop data based on the group properties.
For example, we might want to keep all groups in which the standard deviation is larger than some critical value:

In [47]:
def filter_func(x):
    return x['data2'].mean() > 4.1

display('df', "df.groupby('key').mean()",
        "df.groupby('key').filter(filter_func)")

,key,data1,data2
1,C,0,5
2,B,1,9
3,B,2,2
4,A,3,8
5,A,4,6
6,A,5,0
7,A,6,3
8,A,7,8
9,A,8,5
10,C,9,0


The filter function should return a Boolean value specifying whether the group passes the filtering. Here, because group A does not have a standard deviation greater than 4, it is dropped from the result.

#### Transformation

While aggregation must return a reduced version of the data, transformation can return some transformed version of the full data to recombine.
For such a transformation, the output is the same shape as the input.
A common example is to center the data by subtracting the group-wise mean:

In [63]:
#def center(x):
    #return x - x.mean()
#df_trans = df.groupby('key').transform(center)
df_trans = df.groupby('key').transform(lambda x: (x - x.mean())/x.std())
#display('df_trans', 'df_trabs')
df_trans['key'] = df['key']
df_trans[['key', 'data1', 'data2']].groupby('key').aggregate(['mean','std'])

data1              data2     
             mean  std          mean  std
key                                      
A    0.000000e+00  1.0  0.000000e+00  1.0
B    9.516197e-17  1.0  1.011096e-16  1.0
C   -1.332268e-16  1.0  0.000000e+00  1.0

#### The apply method

The `apply` method lets you apply an arbitrary function to the group results.
The function should take a `DataFrame` and returns either a Pandas object (e.g., `DataFrame`, `Series`) or a scalar; the behavior of the combine step will be tailored to the type of output returned.

For example, here is an `apply` operation that normalizes the first column by the sum of the second:

In [68]:
def norm_by_data2(x):
    # x is a DataFrame of group values
    x['data1'] /= x['data2'].sum()
    return x

ddf = df.groupby('key').apply(norm_by_data2, include_groups=False)
ddf

data1  data2
key                    
A   4   0.100000      8
    5   0.133333      6
    6   0.166667      0
    7   0.200000      3
    8   0.233333      8
    9   0.266667      5
B   2   0.031250      9
    3   0.062500      2
    11  0.312500      7
    13  0.375000      8
    14  0.406250      1
    17  0.500000      0
    18  0.531250      5
C   1   0.000000      5
    10  0.450000      0
    12  0.550000      7
    15  0.700000      0
    16  0.750000      8

In [69]:
ddf.loc[('A',4)]

,A
,4
data1,0.1
data2,8.0


`apply` within a `GroupBy` is flexible: the only criterion is that the function takes a `DataFrame` and returns a Pandas object or scalar. What you do in between is up to you!

### Specifying the Split Key

In the simple examples presented before, we split the `DataFrame` on a single column name.
This is just one of many options by which the groups can be defined, and we'll go through some other options for group specification here.

#### A list, array, series, or index providing the grouping keys

The key can be any series or list with a length matching that of the `DataFrame`. For example:

In [71]:
L = rng.integers(0, 3, len(key))
display('df', "df.groupby(L).sum()")

,key,data1,data2
1,C,0,5
2,B,1,9
3,B,2,2
4,A,3,8
5,A,4,6
6,A,5,0
7,A,6,3
8,A,7,8
9,A,8,5
10,C,9,0


Of course, this means there's another, more verbose way of accomplishing the `df.groupby('key')` from before:

In [74]:
df.groupby(df['key']).sum()

,data1,data2
key,,
A,33,30
B,71,32
C,49,20


#### A dictionary or series mapping index to group

Another method is to provide a dictionary that maps index values to the group keys:

In [75]:
df2 = df.set_index('key')
mapping = {'A': 'vowel', 'B': 'consonant', 'C': 'consonant'}
display('df2', 'df2.groupby(mapping).sum()')

,data1,data2
key,,
C,0,5
B,1,9
B,2,2
A,3,8
A,4,6
A,5,0
A,6,3
A,7,8
A,8,5


#### Any Python function

Similar to mapping, you can pass any Python function that will input the index value and output the group:

In [76]:
df2.groupby(str.lower).mean()

,data1,data2
key,,
a,5.500000,5.000000
b,10.142857,4.571429
c,9.800000,4.000000


#### A list of valid keys

Further, any of the preceding key choices can be combined to group on a multi-index:

In [77]:
df2.groupby([str.lower, mapping]).mean()

,,data1,data2
key,key,,
a,vowel,5.500000,5.000000
b,consonant,10.142857,4.571429
c,consonant,9.800000,4.000000


### Grouping Example

As an example of this, in a few lines of Python code we can put all these together and count discovered planets by method and by decade:

In [78]:
decade = 10 * (planets['year'] // 10)
decade = decade.astype(str) + 's'
decade.name = 'decade'
planets.groupby(['method', decade])['number'].sum().fillna(0)

method                         decade
Astrometry                     2010s       2
Eclipse Timing Variations      2000s       5
                               2010s      10
Imaging                        2000s      29
                               2010s      21
Microlensing                   2000s      12
                               2010s      15
Orbital Brightness Modulation  2010s       5
Pulsar Timing                  1990s       9
                               2000s       1
                               2010s       1
Pulsation Timing Variations    2000s       1
Radial Velocity                1980s       1
                               1990s      52
                               2000s     475
                               2010s     424
Transit                        2000s      64
                               2010s     712
Transit Timing Variations      2010s       9
Name: number, dtype: int64

This shows the power of combining many of the operations we've discussed up to this point when looking at realistic datasets: we quickly gain a coarse understanding of when and how extrasolar planets were detected in the years after the first discovery.

I would suggest digging into these few lines of code and evaluating the individual steps to make sure you understand exactly what they are doing to the result.
It's certainly a somewhat complicated example, but understanding these pieces will give you the means to similarly explore your own data.